# 🛰️ Pista B — El satélite: qué pasó y a quién le pasó
**Código que Salva Vidas · DevFest Santo Domingo 2026 · Taller · Nivel principiante**

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alcastaro/codigo-que-salva-vidas/blob/main/taller/pista_B_satelite.ipynb)

Melissa pasó sobre **Black River**, en el suroeste de Jamaica, el 28 de octubre de 2025. En esta pista usas imágenes de satélite **gratuitas y abiertas** para responder tres preguntas:

| Paso | Pregunta | Herramienta |
|---|---|---|
| 1 | ¿Qué imágenes hay de antes y de después? | Catálogos **STAC** de Sentinel |
| 2 | ¿Qué le pasó a la vegetación? | **Sentinel-2** (óptico) |
| 3 | ¿Dónde se acumuló agua, aunque hubiera nubes? | **Sentinel-1** (radar) |
| 4 | ¿Había gente viviendo ahí? | **AlphaEarth Foundations** (Google DeepMind) |
| 5 | ¿Cómo se lo contamos a la población? | **Gemini** |

Todo esto **no requiere cuenta ni proyecto de Google Cloud**: los datos son abiertos.
Ejecuta cada celda con ▶️ en orden.

In [ ]:
# Instala lo que falta en Colab y trae las funciones de apoyo del taller (~30 s)
!pip -q install rasterio pystac-client planetary-computer google-genai
!wget -q -N https://raw.githubusercontent.com/alcastaro/codigo-que-salva-vidas/main/taller/ayuda.py
import ayuda, numpy as np, pandas as pd, matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "font.size": 10})
print("Listo ✅")

In [ ]:
# @title La zona de estudio { run: "auto" }
LAT, LON = 18.04, -77.82   # Black River y su humedal (Jamaica)
KM = 10                    # lado de la caja, en kilómetros
bbox = ayuda.caja(LAT, LON, KM)
print("Caja (oeste, sur, este, norte):", [round(x, 3) for x in bbox])

## Paso 1 · Buscar imágenes como un profesional: STAC

**STAC** (*SpatioTemporal Asset Catalog*) es el estándar abierto para buscar imágenes de satélite: le das un lugar y unas fechas, y te devuelve las escenas disponibles.
Usamos dos catálogos públicos:
- **Earth Search** (Element 84, sobre AWS) para **Sentinel-2**: fotos ópticas de 10 m, como una cámara, con las nubes como problema.
- **Microsoft Planetary Computer** para **Sentinel-1**: radar, que **atraviesa las nubes** y funciona de noche.

Los dos satélites son del programa europeo **Copernicus**, y sus datos son libres.

In [ ]:
s2 = ayuda.buscar_escenas("sentinel-2-l2a", bbox, "2025-10-15/2025-11-10", max_nubes=60)
print("Sentinel-2 (óptico):"); display(s2.drop(columns="item"))
s1 = ayuda.buscar_escenas("sentinel-1-rtc", bbox, "2025-10-10/2025-11-05", catalogo=ayuda.PLANETARY)
print("Sentinel-1 (radar):"); display(s1.drop(columns="item"))

**🧠 Para pensar:** quita `max_nubes=60` y vuelve a buscar. Mira las escenas del 25 y el 28 de octubre, los días del huracán: **100 % de nubes**. Por eso necesitamos el radar.

## Paso 2 · Antes y después en color real

Elegimos una fecha de antes (**20-oct**) y otra de después (**30-oct**, dos días tras el huracán). Black River queda en el borde entre dos *tiles* (cuadrículas) de Sentinel-2, así que unimos las dos en un **mosaico**.
El **NDVI** mide qué tan verde y sana está la vegetación, usando la luz roja y el infrarrojo cercano que las plantas reflejan.

In [ ]:
# @title Antes y después { run: "auto" }
EN_VIVO = False  # @param {type:"boolean"}
# EN_VIVO = True lee las imágenes ahora mismo desde AWS (30–60 s). False usa una copia idéntica guardada en GitHub.
if EN_VIVO:
    antes   = list(s2[s2.fecha.str.startswith("2025-10-20")].item)
    despues = list(s2[s2.fecha.str.startswith("2025-10-30")].item)
    A, D = ayuda.sentinel2(antes, bbox), ayuda.sentinel2(despues, bbox)
    rgb_a, rgb_d, perdida = A["rgb"], D["rgb"], A["ndvi"] - D["ndvi"]
else:
    copia = ayuda.copia_sentinel()
    rgb_a, rgb_d = copia["rgb_antes"], copia["rgb_despues"]
    perdida = copia["ndvi_antes"] - copia["ndvi_despues"]

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(rgb_a); ax[0].set_title("20 de octubre (antes)")
ax[1].imshow(rgb_d); ax[1].set_title("30 de octubre (después)")
im = ax[2].imshow(perdida, cmap="RdYlGn_r", vmin=-0.4, vmax=0.4); ax[2].set_title("Pérdida de vegetación (NDVI)")
plt.colorbar(im, ax=ax[2], shrink=0.7)
for a in ax: a.axis("off")
plt.show()

## Paso 3 · El radar ve el agua a través de las nubes

El radar de Sentinel-1 lanza microondas y mide cuánto rebota. El **agua quieta** actúa como un espejo y devuelve muy poco, así que sale **oscura**.
Regla simple: si un píxel pasa de "claro" antes a "oscuro" después, ahí apareció **agua nueva**.
Comparamos dos pasadas de la **misma órbita** (17-oct y 29-oct) para que la geometría sea idéntica.

In [ ]:
# @title El radar antes y después { run: "auto" }
EN_VIVO = False  # @param {type:"boolean"}
# EN_VIVO = True lee el radar desde Microsoft Planetary Computer (de 20 s a varios minutos, según la red).
if EN_VIVO:
    orbita = 150
    r_antes   = s1[(s1.orbita == orbita) & s1.fecha.str.startswith("2025-10-17")].item.iloc[0]
    r_despues = s1[(s1.orbita == orbita) & s1.fecha.str.startswith("2025-10-29")].item.iloc[0]
    db_a, db_d = ayuda.radar_db(r_antes, bbox), ayuda.radar_db(r_despues, bbox)
else:
    copia = ayuda.copia_sentinel()
    db_a, db_d = copia["db_antes"], copia["db_despues"]

UMBRAL_DB = -18   # por debajo de esto, lo tratamos como agua
agua_nueva = (db_d < UMBRAL_DB) & ~(db_a < UMBRAL_DB)
km2 = agua_nueva.sum() * ayuda.km2_por_pixel(bbox, agua_nueva.shape)

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(db_a, cmap="gray", vmin=-25, vmax=0); ax[0].set_title("Radar 17-oct")
ax[1].imshow(db_d, cmap="gray", vmin=-25, vmax=0); ax[1].set_title("Radar 29-oct")
ax[2].imshow(db_d, cmap="gray", vmin=-25, vmax=0)
ax[2].imshow(np.ma.masked_where(~agua_nueva, agua_nueva), cmap="cool", alpha=0.9)
ax[2].set_title(f"Agua nueva ≈ {km2:.1f} km²")
for a in ax: a.axis("off")
plt.show()

**🧠 Para pensar:** ¿ves "agua nueva" en el **mar**? No es una inundación. Después del huracán había viento, el mar se picó, y eso cambia el rebote del radar.
Un umbral simple tiene límites. Los sistemas reales, como Flood Hub, usan modelos entrenados y máscaras de agua permanente.

## Paso 4 · ¿Qué había en el suelo? AlphaEarth Foundations

**AlphaEarth Foundations** (Google DeepMind) resume años de observaciones de satélite (óptico, radar, relieve, clima…) en **64 números por cada píxel de 10 m**. Esos 64 números son un **embedding**, una especie de "huella digital" del terreno.
Dos lugares parecidos (dos barrios, dos manglares) tienen huellas parecidas. Así se puede **buscar por similitud** sin entrenar ningún modelo.

Los embeddings anuales son abiertos en **Source Cooperative** (CC BY 4.0). Para Black River usamos una **copia** de 2024 (el año anterior al huracán), porque leer el original tarda unos minutos.
Al final puedes probar con tu propio lugar.

In [ ]:
emb = ayuda.cargar_embeddings("data/alphaearth/black_river_2024.npz")
print("Forma:", emb.shape, "→ 64 números por píxel")

fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].imshow(rgb_a); ax[0].set_title("Sentinel-2 (lo que ve una cámara)")
ax[1].imshow(ayuda.rgb_embeddings(emb)); ax[1].set_title("AlphaEarth 2024 (3 de 64 dimensiones como color)")
for a in ax: a.axis("off")
plt.show()

Ahora le damos a AlphaEarth un ejemplo de **zona construida** (el centro de Black River) y le pedimos que encuentre todo lo que se parece.
Después **cruzamos** esa capa con el agua nueva del radar: **zonas construidas a menos de 250 m de agua nueva**. Así se mide la *exposición* en una evaluación rápida.

In [ ]:
# @title Busca lugares parecidos a este punto { run: "auto" }
ref_lat, ref_lon = 18.033, -77.858   # centro del pueblo de Black River
UMBRAL_SIM = 0.90  # @param {type:"slider", min:0.6, max:0.98, step:0.01}

v = ayuda.vector_en(emb, bbox, ref_lat, ref_lon)
sim = ayuda.similitud(emb, v)
construido = sim > UMBRAL_SIM

agua_en_emb = ayuda.redimensionar(agua_nueva, construido.shape)   # misma cuadrícula que AlphaEarth
expuesto = construido & ayuda.cerca_de(agua_en_emb, bbox, metros=250)
km2_px = ayuda.km2_por_pixel(bbox, construido.shape)

fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].imshow(sim, cmap="viridis", vmin=0.5, vmax=1); ax[0].set_title("Parecido al pueblo (1 = idéntico)")
ax[1].imshow(rgb_d); ax[1].set_title(f"Construido a <250 m de agua nueva ≈ {expuesto.sum() * km2_px:.2f} km²")
capa = ayuda.redimensionar(expuesto, rgb_d.shape[:2])
ax[1].imshow(np.ma.masked_where(~capa, capa), cmap="autumn", alpha=0.9)
for a in ax: a.axis("off")
plt.show()
print(f"Zona 'parecida a pueblo': {construido.sum() * km2_px:.2f} km² · de ella, junto a agua nueva: {expuesto.sum() * km2_px:.2f} km²")

### 🔑 Tu API key de Gemini (2 minutos, solo la primera vez)
1. Abre **[aistudio.google.com/apikey](https://aistudio.google.com/apikey)** y pulsa **Create API key**. Es gratis.
2. En Colab, abre el panel de la **llave 🔑 (Secretos)** a la izquierda.
3. Pulsa **Add new secret**: nombre `GOOGLE_API_KEY`, valor tu key, y activa **Notebook access**.

> Atajo: en ese mismo panel, **Gemini API keys → Import key from Google AI Studio** lo hace por ti.
>
> **Nunca pegues una key dentro de una celda.** Si compartes el notebook o lo subes a GitHub, la key viaja con él.

## Paso 5 · De los mapas a una alerta
Le pasamos los números a **Gemini** para que redacte un aviso claro: qué pasó, dónde y qué hacer.

In [ ]:
cliente = ayuda.cliente_gemini()
modelo = ayuda.elegir_modelo(cliente, "flash")
datos = {
    "evento": "Huracán Melissa, inundación posterior",
    "zona": "Black River y alrededores, parroquia de St. Elizabeth, Jamaica",
    "fuente": "Satélites Sentinel-1 (radar, 29-oct-2025) y Sentinel-2, análisis automático",
    "agua_nueva_km2": round(float(km2), 1),
    "zona_construida_a_menos_de_250m_de_agua_nueva_km2": round(float(expuesto.sum() * km2_px), 2),
    "vegetacion": "pérdida fuerte de vegetación tras el huracán",
    "advertencia": "Estimación automática por satélite, sin validar en terreno",
}
ayuda.mostrar_alerta(ayuda.redactar_alerta(cliente, modelo, datos))

## 🏁 Reto · Tu propio lugar
Cambia las coordenadas por tu barrio, el río cerca de tu casa o tu provincia. Usa una caja **pequeña** (3 km): leer AlphaEarth en vivo tarda **alrededor de un minuto por año**.
Compara 2017 con 2025: ¿qué cambió en ocho años? Busca, por ejemplo, la orilla del Ozama o la expansión de Santiago.

In [ ]:
# @title Compara dos años en tu lugar
mi_lat, mi_lon = 18.4861, -69.9312  # @param {type:"number"}
anio_1, anio_2 = 2017, 2025          # @param {type:"raw"}
mi_bbox = ayuda.caja(mi_lat, mi_lon, km=3)

e1 = ayuda.embeddings_alphaearth(mi_bbox, anio_1, ancho=300)
e2 = ayuda.embeddings_alphaearth(mi_bbox, anio_2, ancho=300)
cambio = 1 - np.nansum(e1 * e2, 0) / (np.sqrt(np.nansum(e1**2, 0)) * np.sqrt(np.nansum(e2**2, 0)))

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(ayuda.rgb_embeddings(e1)); ax[0].set_title(str(anio_1))
ax[1].imshow(ayuda.rgb_embeddings(e2)); ax[1].set_title(str(anio_2))
im = ax[2].imshow(cambio, cmap="magma", vmin=0, vmax=0.6); ax[2].set_title("Cuánto cambió (claro = mucho)")
plt.colorbar(im, ax=ax[2], shrink=0.7)
for a in ax: a.axis("off")
plt.show()

---
**Fuentes y licencias.** Sentinel-1 y Sentinel-2: Copernicus (ESA/UE), datos libres; Sentinel-2 L2A vía Earth Search (Element 84) y Sentinel-1 RTC vía Microsoft Planetary Computer.
AlphaEarth Foundations Satellite Embedding: Google DeepMind / Google Earth Engine, copia en Source Cooperative (CC BY 4.0).
Los análisis de este notebook son **didácticos**: un umbral de radar no sustituye una evaluación de daños oficial.